
# Topic 4 — Preference Optimization + RL (SFT → DPO → GRPO)

**Core question:** *How do I make outputs more preferred, or improve reasoning?*

**New running example for this notebook**: a simple multi-step math word
problem task, which makes "correct vs incorrect" unambiguous -- essential
for demonstrating a verifiable reward function in GRPO.

```mermaid
flowchart LR
    A[Base model] --> B["SFT<br/>learn the task format +<br/>correct reasoning style"]
    B --> C["DPO<br/>learn to prefer GOOD solutions<br/>over BAD ones"]
    C --> D["GRPO<br/>generate several candidates,<br/>reward the correct ones"]
```

**The progression, in one line each:**
- **SFT**: teach the model *what a correct, step-by-step solution looks like*.
- **DPO**: given a *preferred* and a *rejected* solution for the same
  problem, teach the model to raise the preferred one's probability and
  lower the rejected one's.
- **GRPO**: for each problem, have the model generate *several* candidate
  solutions, score each with a **verifier** (reward = 1 if the final answer
  is correct, else 0), and update the model to favor whatever it did that
  led to higher-reward candidates.

### Can you answer this?
1. What does DPO need that plain SFT doesn't?
2. What does GRPO need that DPO doesn't?
3. Why is a *verifiable* reward (like "is the final numeric answer correct")
   such a good fit for GRPO?


In [1]:

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()
def banner(title, subtitle="", style="bold white on blue"):
    text = title if not subtitle else f"{title}\n[dim]{subtitle}[/dim]"
    console.print(Panel(text, style=style, expand=False, box=box.DOUBLE))
def section(title):
    console.print(f"\n[bold cyan]── {title} " + "─" * max(0, 60 - len(title)) + "[/bold cyan]")
def kv_table(title, rows, caption=""):
    table = Table(title=title, box=box.SIMPLE_HEAVY, caption=caption)
    table.add_column("Key", style="bold")
    table.add_column("Value")
    for k, v in rows.items():
        table.add_row(str(k), str(v))
    console.print(table)

banner("Topic 4: Preference Optimization + RL", "SFT -> DPO -> GRPO on a math reasoning task")


╔═════════════════════════════════════════════╗
║ Topic 4: Preference Optimization + RL       ║
║ SFT -> DPO -> GRPO on a math reasoning task ║
╚═════════════════════════════════════════════╝


## Concept primer

**DPO (Direct Preference Optimization)**: each training example is a triple
`(prompt, preferred_response, rejected_response)`. DPO's loss directly
increases the model's relative preference for `preferred_response` over
`rejected_response`, without needing a separate reward model (unlike
classic RLHF) -- it's "direct" because the preference signal is baked
straight into the loss.

**GRPO (Group Relative Policy Optimization)**: for each prompt, the
*current* model generates a *group* of several candidate completions. A
**reward function** (here: a verifier checking the final numeric answer)
scores each candidate. The model is updated to make the higher-reward
candidates within the group more likely relative to the lower-reward ones
-- "group relative" because rewards are compared *within* each prompt's
group of candidates, not against a fixed external baseline.

**Verifier / reward function**: a plain function, `reward = 1 if correct
else 0`, that doesn't require training a separate reward model. This is
only possible when correctness is mechanically checkable (math, code
execution, structured outputs) -- which is exactly why math/code tasks are
GRPO's classic playground.

**Why this progression (SFT -> DPO -> GRPO) rather than jumping straight to
GRPO**: GRPO needs the model to already produce *somewhat* reasonable
candidates for the reward signal to be informative (if every candidate is
garbage, there's nothing useful to compare). SFT gets the model to a
reasonable starting point; DPO sharpens preference between specific
good/bad pairs; GRPO then explores and reinforces correctness directly
against a ground-truth checkable signal.


In [2]:

import torch
section("Environment check")
HAS_CUDA = torch.cuda.is_available()
kv_table("Runtime", {"CUDA available": HAS_CUDA, "PyTorch": torch.__version__})
if not HAS_CUDA:
    console.print(Panel(
        "[bold yellow]No CUDA GPU detected.[/bold yellow] Dataset/preference-pair construction and "
        "the verifier function below are real CPU-safe work. The three actual training runs "
        "(SFT, DPO, GRPO) require a GPU -- run on Colab (T4) for those.",
        title="GPU NOT AVAILABLE HERE", style="yellow", box=box.HEAVY,
    ))


── Environment check ───────────────────────────────────────────

          Runtime          
                           
  Key              Value   
 ━━━━━━━━━━━━━━━━━━━━━━━━━ 
  CUDA available   False   
  PyTorch          2.14.1

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ GPU NOT AVAILABLE HERE ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ No CUDA GPU detected. Dataset/preference-pair construction and the verifier function below are real CPU-safe    ┃
┃ work. The three actual training runs (SFT, DPO, GRPO) require a GPU -- run on Colab (T4) for those.             ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

In [3]:

import sys
IN_COLAB = "google.colab" in sys.modules
packages = ["unsloth", "transformers", "trl", "peft", "bitsandbytes", "torch", "datasets", "rich"]
if IN_COLAB:
    get_ipython().system(f"pip install -q {' '.join(packages)}")
    console.print("[green]Installed packages for Colab.[/green]")
else:
    console.print("[dim]Not running in Colab -- assuming the project virtualenv already has these.[/dim]")


Not running in Colab -- assuming the project virtualenv already has these.

In [4]:
# --- Load the math problem dataset ------------------------------------------
import json
import random
from pathlib import Path

section("Loading the math problem dataset")

vsc_ipynb_file = globals().get("__vsc_ipynb_file__")
search_roots = [Path.cwd()]
if vsc_ipynb_file:
    search_roots.insert(0, Path(vsc_ipynb_file).resolve().parent)
search_roots += [root.parent for root in list(search_roots)]

RELATIVE_CANDIDATES = ["data/math_problems.json", "04_preference_optimization_rl/data/math_problems.json"]
DATA_PATH = None
for root in search_roots:
    for rel in RELATIVE_CANDIDATES:
        candidate = root / rel
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH:
        break

if DATA_PATH is None:
    # Self-sufficient fallback: regenerate the exact same deterministic dataset
    # in-memory instead of requiring a separate file upload (e.g. on Colab).
    console.print("[dim]math_problems.json not found nearby -- generating it in-memory "
                  "instead of requiring a separate file upload.[/dim]")

    NAMES = ["Sarah", "Alex", "Jordan", "Priya", "Sam", "Taylor", "Morgan", "Chris"]
    ITEMS = ["apples", "boxes of pens", "notebooks", "stickers", "marbles", "coins"]

    def _make_problem(rng):
        name = rng.choice(NAMES)
        item = rng.choice(ITEMS)
        groups = rng.randint(2, 6)
        per_group = rng.randint(3, 12)
        given_away = rng.randint(1, groups * per_group - 1)
        total = groups * per_group
        final_answer = total - given_away
        problem = (f"{name} has {groups} groups of {item}, with {per_group} {item} in each group. "
                   f"{name} gives away {given_away} {item}. How many {item} does {name} have left?")
        correct_solution = (
            f"Step 1: {name} starts with {groups} groups of {per_group} {item}, "
            f"so the total is {groups} x {per_group} = {total} {item}.\n"
            f"Step 2: {name} gives away {given_away} {item}, so {total} - {given_away} = {final_answer} {item}.\n"
            f"#### {final_answer}")
        wrong_answer = total
        wrong_solution = (
            f"Step 1: {name} has {groups} groups of {per_group} {item}, "
            f"so the total is {groups} x {per_group} = {total} {item}.\n"
            f"Step 2: {name} has {total} {item} left.\n"
            f"#### {wrong_answer}")
        return {"problem": problem, "correct_solution": correct_solution,
                "wrong_solution": wrong_solution, "final_answer": final_answer}

    def generate_math_problems(n=200, seed=42):
        rng = random.Random(seed)
        return [_make_problem(rng) for _ in range(n)]

    problems = generate_math_problems()
    DATA_PATH = Path.cwd() / "data" / "math_problems.json"
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    with DATA_PATH.open("w") as f:
        json.dump(problems, f, indent=2)
else:
    with DATA_PATH.open() as f:
        problems = json.load(f)

console.print(f"Loaded [bold]{len(problems)}[/bold] math problems from {DATA_PATH}")
console.print(Panel(
    f"[bold]Problem:[/bold] {problems[0]['problem']}\n\n"
    f"[bold]Correct solution:[/bold]\n{problems[0]['correct_solution']}\n\n"
    f"[bold]Wrong solution (used as the DPO 'rejected' example):[/bold]\n{problems[0]['wrong_solution']}",
    title="Sample problem", box=box.ROUNDED,
))


── Loading the math problem dataset ────────────────────────────

Loaded 200 math problems from 
/Users/bhanupraka.gopularam/aiprojects/llm-fine-tuning/04_preference_optimization_rl/data/math_problems.json

╭──────────────────────────────────────────────── Sample problem ─────────────────────────────────────────────────╮
│ Problem: Alex has 4 groups of apples, with 6 apples in each group. Alex gives away 8 apples. How many apples    │
│ does Alex have left?                                                                                            │
│                                                                                                                 │
│ Correct solution:                                                                                               │
│ Step 1: Alex starts with 4 groups of 6 apples, so the total is 4 x 6 = 24 apples.                               │
│ Step 2: Alex gives away 8 apples, so 24 - 8 = 16 apples.                                                        │
│ #### 16                                                                                                         │
│                                                                                                                 │
│ Wrong solution (used as the DPO 'rejected' example):                                                            │
│ Step 1: Alex has 4 groups of 6 apples, so the total is 4 x 6 = 24 apples.                                       │
│ Step 2: Alex has 24 apples left.                                                                                │
│ #### 24                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Stage 1 — SFT: learn the task format + correct reasoning

We SFT on `(problem -> correct_solution)` pairs, exactly like Topic 1 --
just with a different task. The target format (`Step 1: ... Step 2: ... ####
<answer>`) is itself something the model needs to learn to produce
consistently, since the `####` marker is what our verifier will look for
later in GRPO.


In [5]:

# --- SFT dataset construction (CPU-safe) + training (GPU-only) -------------
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a careful math tutor. Solve the problem step by step, then give the "
    "final numeric answer on its own line as '#### <number>'."
)

def to_sft_conversation(item):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["problem"]},
        {"role": "assistant", "content": item["correct_solution"]},
    ]

import random
random.seed(7)
shuffled = problems[:]
random.shuffle(shuffled)
N_HOLDOUT = 10
holdout = shuffled[:N_HOLDOUT]
train_problems = shuffled[N_HOLDOUT:]

MODEL_NAME = "Qwen/Qwen3-0.6B"
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_text(item):
    return {"text": tokenizer.apply_chat_template(to_sft_conversation(item), tokenize=False)}

sft_dataset = Dataset.from_list([to_text(p) for p in train_problems])
console.print(f"SFT training set ready: {sft_dataset}")

if HAS_CUDA:
    from unsloth import FastLanguageModel
    from trl import SFTTrainer, SFTConfig

    model, tok = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=512)
    FastLanguageModel.for_training(model)
    cfg = SFTConfig(output_dir="outputs/sft_math", num_train_epochs=3,
                     per_device_train_batch_size=4, gradient_accumulation_steps=2,
                     learning_rate=2e-4, max_length=512, logging_steps=20,
                     save_strategy="no", report_to=[])
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=sft_dataset, args=cfg)
    section("Stage 1: SFT training")
    sft_result = trainer.train()
    kv_table("SFT result", {"final loss": sft_result.training_loss})
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- SFT training requires a CUDA GPU.\n"
        "Expected on Colab: training loss dropping over ~3 epochs as the model learns the\n"
        "Step 1/Step 2/#### <answer> format consistently.",
        title="Stage 1: SFT training (GPU-only)", style="yellow",
    ))


SFT training set ready: Dataset({
    features: ['text'],
    num_rows: 190
})

╭─────────────────────────────────────── Stage 1: SFT training (GPU-only) ────────────────────────────────────────╮
│ SKIPPED -- SFT training requires a CUDA GPU.                                                                    │
│ Expected on Colab: training loss dropping over ~3 epochs as the model learns the                                │
│ Step 1/Step 2/#### <answer> format consistently.                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Stage 2 — DPO: prefer the correct solution over the plausible wrong one

Each training example pairs the **same problem** with a **preferred**
(correct) and **rejected** (wrong, but plausible-looking) solution --
exactly the `correct_solution` / `wrong_solution` fields we generated.


In [6]:

# --- DPO dataset construction (CPU-safe) + training (GPU-only) -------------
def to_dpo_example(item):
    prompt_conv = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item["problem"]}]
    prompt_text = tokenizer.apply_chat_template(prompt_conv, tokenize=False, add_generation_prompt=True)
    return {
        "prompt": prompt_text,
        "chosen": item["correct_solution"],
        "rejected": item["wrong_solution"],
    }

dpo_dataset = Dataset.from_list([to_dpo_example(p) for p in train_problems])
console.print(f"DPO training set ready: {dpo_dataset}")
console.print(Panel(
    f"[bold]Prompt (truncated):[/bold] ...{dpo_dataset[0]['prompt'][-120:]}\n\n"
    f"[bold green]Chosen:[/bold green] {dpo_dataset[0]['chosen'][:100]}...\n\n"
    f"[bold red]Rejected:[/bold red] {dpo_dataset[0]['rejected'][:100]}...",
    title="Sample DPO training example", box=box.ROUNDED,
))

if HAS_CUDA:
    from trl import DPOTrainer, DPOConfig

    dpo_cfg = DPOConfig(output_dir="outputs/dpo_math", num_train_epochs=2,
                          per_device_train_batch_size=2, learning_rate=5e-6,
                          logging_steps=20, save_strategy="no", report_to=[])
    dpo_trainer = DPOTrainer(model=model, args=dpo_cfg, train_dataset=dpo_dataset, processing_class=tok)
    section("Stage 2: DPO training")
    dpo_result = dpo_trainer.train()
    kv_table("DPO result", {"final loss": dpo_result.training_loss})
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- DPO training requires a CUDA GPU (and the SFT model from Stage 1).\n"
        "Expected on Colab: DPO loss decreasing as the model's relative preference for the\n"
        "chosen (correct) solution over the rejected one grows.",
        title="Stage 2: DPO training (GPU-only)", style="yellow",
    ))


DPO training set ready: Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 190
})

╭────────────────────────────────────────── Sample DPO training example ──────────────────────────────────────────╮
│ Prompt (truncated): ...apples in each group. Chris gives away 26 apples. How many apples does Chris have        │
│ left?<|im_end|>                                                                                                 │
│ <|im_start|>assistant                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│ Chosen: Step 1: Chris starts with 5 groups of 8 apples, so the total is 5 x 8 = 40 apples.                      │
│ Step 2: Chris giv...                                                                                            │
│                                                                                                                 │
│ Rejected: Step 1: Chris has 5 groups of 8 apples, so the total is 5 x 8 = 40 apples.                            │
│ Step 2: Chris has 40 appl...                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── Stage 2: DPO training (GPU-only) ────────────────────────────────────────╮
│ SKIPPED -- DPO training requires a CUDA GPU (and the SFT model from Stage 1).                                   │
│ Expected on Colab: DPO loss decreasing as the model's relative preference for the                               │
│ chosen (correct) solution over the rejected one grows.                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Stage 3 — GRPO: reward correctness directly with a verifier

Instead of fixed chosen/rejected pairs, GRPO has the model generate several
of its *own* candidate solutions per problem, and a **verifier function**
scores each one. Let's build and test that verifier first -- this part is
pure Python and fully real, no GPU needed.


In [7]:

# --- The verifier / reward function (fully CPU-safe, tested on real cases) --
import re

def extract_final_answer(text):
    match = re.search(r"####\s*(-?\d+)", text)
    if match:
        return int(match.group(1))
    return None

def verifier_reward(completion_text, correct_answer):
    predicted = extract_final_answer(completion_text)
    return 1.0 if predicted == correct_answer else 0.0

section("Testing the verifier function on real sample completions")
test_cases = [
    ("Correct completion", problems[0]["correct_solution"], problems[0]["final_answer"]),
    ("Wrong completion", problems[0]["wrong_solution"], problems[0]["final_answer"]),
    ("Malformed completion (no #### marker)", "I think the answer is sixteen.", problems[0]["final_answer"]),
]
table = Table(title="Verifier reward -- real test cases", box=box.SIMPLE_HEAVY, show_lines=True)
table.add_column("Case")
table.add_column("Extracted answer")
table.add_column("Correct answer")
table.add_column("Reward", style="bold")
for name, text, correct in test_cases:
    extracted = extract_final_answer(text)
    reward = verifier_reward(text, correct)
    table.add_row(name, str(extracted), str(correct), f"{reward:.1f}")
console.print(table)


── Testing the verifier function on real sample completions ────

                          Verifier reward -- real test cases                          
                                                                                      
  Case                                    Extracted answer   Correct answer   Reward  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Correct completion                      16                 16               1.0     
                                                                                      
  Wrong completion                        24                 16               0.0     
                                                                                      
  Malformed completion (no #### marker)   None               16               0.0    

In [8]:

# --- GRPO config + training (GPU-only) --------------------------------------
def reward_fn(prompts, completions, **kwargs):
    # TRL's GRPOTrainer calls the reward function with the batch of generated
    # completions; we look up each prompt's correct answer and score accordingly.
    answers = kwargs.get("final_answer", [None] * len(completions))
    return [verifier_reward(c, a) for c, a in zip(completions, answers)]

if HAS_CUDA:
    from trl import GRPOTrainer, GRPOConfig

    def to_grpo_example(item):
        prompt_conv = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item["problem"]}]
        return {
            "prompt": tokenizer.apply_chat_template(prompt_conv, tokenize=False, add_generation_prompt=True),
            "final_answer": item["final_answer"],
        }
    grpo_dataset = Dataset.from_list([to_grpo_example(p) for p in train_problems])

    grpo_cfg = GRPOConfig(output_dir="outputs/grpo_math", num_train_epochs=1,
                            per_device_train_batch_size=2, num_generations=4,
                            learning_rate=1e-6, logging_steps=20, save_strategy="no", report_to=[])
    grpo_trainer = GRPOTrainer(model=model, args=grpo_cfg, train_dataset=grpo_dataset,
                                 reward_funcs=reward_fn, processing_class=tok)
    section("Stage 3: GRPO training")
    grpo_result = grpo_trainer.train()
    kv_table("GRPO result", {"final loss": grpo_result.training_loss})
else:
    console.print(Panel(
        "[yellow]SKIPPED[/yellow] -- GRPO training requires a CUDA GPU (and the DPO model from Stage 2).\n"
        "Expected on Colab: for each prompt, 4 candidate completions are generated; the\n"
        "fraction scoring reward=1 (correct final answer) should trend upward over training\n"
        "as the model learns to favor whatever reasoning patterns reliably produce correct answers.",
        title="Stage 3: GRPO training (GPU-only)", style="yellow",
    ))


╭─────────────────────────────────────── Stage 3: GRPO training (GPU-only) ───────────────────────────────────────╮
│ SKIPPED -- GRPO training requires a CUDA GPU (and the DPO model from Stage 2).                                  │
│ Expected on Colab: for each prompt, 4 candidate completions are generated; the                                  │
│ fraction scoring reward=1 (correct final answer) should trend upward over training                              │
│ as the model learns to favor whatever reasoning patterns reliably produce correct answers.                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


## Can you answer this? (Topic 4 training checkpoints)

**1. What does DPO need that plain SFT doesn't?**
<details><summary>Answer</summary>
A *pair* of responses per prompt -- a preferred one and a rejected one --
rather than a single target response. DPO's loss is defined entirely in
terms of the relative probability the model assigns to the preferred vs.
rejected response.
</details>

**2. What does GRPO need that DPO doesn't?**
<details><summary>Answer</summary>
A reward function (verifier) that can score any arbitrary model-generated
candidate, plus the ability to sample multiple candidates per prompt from
the model itself during training. DPO only ever sees the two fixed
responses you give it; GRPO explores the model's own generation space.
</details>

**3. Why is a verifiable reward (like "is the final numeric answer correct")
such a good fit for GRPO?**
<details><summary>Answer</summary>
Because it requires no separate reward model and no human labeling at
training time -- it's a cheap, deterministic, always-available signal.
GRPO can generate as many candidates as compute allows and score every
single one instantly and correctly, which is exactly what lets it explore
effectively. Tasks without a mechanically checkable answer (open-ended
writing, subjective quality) don't have this luxury and typically need a
learned reward model or an LLM judge instead.
</details>


In [9]:

section("Scale discussion: preference optimization + RL at ~100 vs ~10k users")
table = Table(title="DPO/GRPO at scale", box=box.SIMPLE_HEAVY, show_lines=True)
table.add_column("Concern")
table.add_column("~100 users", style="cyan")
table.add_column("~10k users", style="magenta")
rows = [
    ("Preference data", "A few hundred hand-curated or synthetic pairs can show real signal",
                        "Needs thousands of real preference pairs, often collected from production feedback or multiple model variants, with quality review"),
    ("GRPO compute cost", "Multiple generations per prompt during training is still cheap for a small model",
                          "Compute scales with (num_generations x batch size x problems) -- a real cost driver that needs budgeting, not just GPU-count"),
    ("Reward function reliability", "A hand-checked verifier is easy to audit",
                                    "Needs monitoring for reward hacking -- cases where the model finds a way to maximize reward without actually solving the task correctly"),
    ("Deployment risk", "Manual review of before/after outputs is enough",
                        "Needs automated regression suites and staged rollout (canary) before trusting a DPO/GRPO-updated model broadly -- preference-tuned models can shift behavior in subtle, hard-to-spot ways"),
]
for c, s, b in rows:
    table.add_row(c, s, b)
console.print(table)


── Scale discussion: preference optimization + RL at ~100 vs ~10k users 

                                                 DPO/GRPO at scale                                                 
                                                                                                                   
  Concern                       ~100 users                                ~10k users                               
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Preference data               A few hundred hand-curated or synthetic   Needs thousands of real preference       
                                pairs can show real signal                pairs, often collected from production   
                                                                          feedback or multiple model variants,     
                                                                          with quality review                      
                                                                                                                   
  GRPO compute cost             Multiple generations per prompt during    Compute scales with (num_generations x   
                                training is still cheap for a small       batch size x problems) -- a real cost    
                                model                                     driver that needs budgeting, not just    
                                                                          GPU-count                                
                                                                                                                   
  Reward function reliability   A hand-checked verifier is easy to        Needs monitoring for reward hacking --   
                                audit                                     cases where the model finds a way to     
                                                                          maximize reward without actually         
                                                                          solving the task correctly               
                                                                                                                   
  Deployment risk               Manual review of before/after outputs     Needs automated regression suites and    
                                is enough                                 staged rollout (canary) before trusting  
                                                                          a DPO/GRPO-updated model broadly --      
                                                                          preference-tuned models can shift        
                                                                          behavior in subtle, hard-to-spot ways   

In [10]:

section("Summary")
console.print(Panel(
    "[bold]What we did:[/bold]\n"
    "  1. Generated a math word-problem dataset with correct and plausible-wrong solutions\n"
    "  2. (On GPU) ran SFT to teach the task format and correct reasoning style\n"
    "  3. Built DPO preference pairs and (on GPU) trained the model to prefer correct over wrong\n"
    "  4. Built and TESTED a real verifier/reward function (no GPU needed)\n"
    "  5. (On GPU) ran GRPO, rewarding the model directly for correct final answers\n\n"
    "[bold]Next: Topic 5 -- Distillation + Model Compression[/bold]\n"
    "A larger teacher model passes its knowledge to a smaller student,\n"
    "trading some capability for much lower inference cost.",
    title="Topic 4 complete", style="bold green", box=box.DOUBLE,
))


── Summary ─────────────────────────────────────────────────────

╔═══════════════════════════════════════════════ Topic 4 complete ════════════════════════════════════════════════╗
║ What we did:                                                                                                    ║
║   1. Generated a math word-problem dataset with correct and plausible-wrong solutions                           ║
║   2. (On GPU) ran SFT to teach the task format and correct reasoning style                                      ║
║   3. Built DPO preference pairs and (on GPU) trained the model to prefer correct over wrong                     ║
║   4. Built and TESTED a real verifier/reward function (no GPU needed)                                           ║
║   5. (On GPU) ran GRPO, rewarding the model directly for correct final answers                                  ║
║                                                                                                                 ║
║ Next: Topic 5 -- Distillation + Model Compression                                                               ║
║ A larger teacher model passes its knowledge to a smaller student,                                               ║
║ trading some capability for much lower inference cost.                                                          ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝